# Retrieval (검색)

대형 언어 모델(LLM)은 강력하지만 두 가지 주요 한계가 있습니다:

- **제한된 컨텍스트**: 전체 코퍼스를 한 번에 처리할 수 없음
- **정적 지식**: 훈련 데이터가 특정 시점에 고정됨

**Retrieval(검색)**은 쿼리 시점에 관련 외부 지식을 가져와서 이러한 문제를 해결합니다. 이것이 **Retrieval-Augmented Generation (RAG)**의 기초입니다: 컨텍스트별 정보로 LLM의 답변을 향상시킵니다.

## RAG의 핵심 개념

RAG는 검색과 생성을 결합하여 정확하고 맥락에 맞는 답변을 생성합니다. 이번 튜토리얼에서는 2-Step RAG, Agentic RAG, Hybrid RAG의 세 가지 아키텍처를 비교하고, 각각의 구현 방법을 살펴봅니다.

**학습 목표**

- 벡터 저장소를 활용한 지식 베이스 구축 방법을 학습합니다.
- 2-Step RAG, Agentic RAG, Hybrid RAG의 차이점과 활용 사례를 이해합니다.
- `create_agent`를 사용한 Agentic RAG 에이전트를 구현합니다.
- LangGraph를 활용하여 쿼리 전처리, 검색 검증, 생성 후 검사를 포함하는 Hybrid RAG를 구현합니다.
- 메타데이터 필터링을 통한 검색 최적화 기법을 익힙니다.

## 사전 준비

튜토리얼을 시작하기 전에 필요한 환경을 설정합니다. API 키를 환경 변수로 로드하고, LangSmith 추적을 활성화하여 실행 과정을 모니터링할 수 있도록 합니다.

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv(override=True)

True

In [2]:
# LangSmith 추적을 설정합니다.
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("LangGraph-RAG")

LangSmith 추적을 시작합니다.
[프로젝트명]
LangGraph-RAG


## 지식 베이스 구축

**지식 베이스**는 검색 중에 사용되는 문서 또는 구조화된 데이터의 저장소입니다. RAG 시스템의 품질은 지식 베이스의 구성 방식에 크게 좌우됩니다.

### 검색 파이프라인

일반적인 검색 워크플로우는 다음과 같습니다:

1. **문서 로드**: 외부 소스에서 데이터 수집
2. **청크로 분할**: 큰 문서를 작은 조각으로 분할
3. **임베딩 생성**: 텍스트를 벡터로 변환
4. **벡터 저장소에 저장**: 벡터를 검색 가능한 데이터베이스에 저장
5. **쿼리 임베딩**: 사용자 질문을 벡터로 변환
6. **검색**: 유사한 벡터 찾기
7. **LLM에 전달**: 검색된 정보로 답변 생성

### 간단한 지식 베이스 생성

샘플 문서를 사용하여 FAISS 기반의 벡터 저장소를 생성합니다. `RecursiveCharacterTextSplitter`로 문서를 적절한 크기로 분할하고, `OpenAIEmbeddings`로 텍스트를 벡터로 변환한 후 FAISS에 저장합니다.

아래 코드는 4개의 샘플 문서를 생성하고, 임베딩 후 벡터 저장소에 저장합니다.

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# 샘플 문서 로드
documents = [
    Document(
        page_content="LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다. 도구, 에이전트, 메모리 등을 제공합니다.",
        metadata={"source": "docs", "topic": "langchain"}
    ),
    Document(
        page_content="LangGraph는 상태 기반 에이전트를 구축하기 위한 라이브러리입니다. LangChain 위에 구축되었습니다.",
        metadata={"source": "docs", "topic": "langgraph"}
    ),
    Document(
        page_content="RAG는 Retrieval-Augmented Generation의 약자로, 외부 지식을 검색하여 LLM 응답을 향상시킵니다.",
        metadata={"source": "docs", "topic": "rag"}
    ),
    Document(
        page_content="벡터 데이터베이스는 임베딩을 저장하고 검색하는 데 특화된 데이터베이스입니다. FAISS, Pinecone 등이 있습니다.",
        metadata={"source": "docs", "topic": "vectordb"}
    ),
]

# 텍스트 분할기 (큰 문서의 경우)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
# 청크 분할
splits = text_splitter.split_documents(documents)

# 임베딩 생성
embeddings = OpenAIEmbeddings()
# 벡터 저장소 저장
vectorstore = FAISS.from_documents(splits, embeddings)

print(f"Created vector store with {len(splits)} documents")

Created vector store with 4 documents


### 벡터 저장소에서 검색

생성한 벡터 저장소에서 유사도 검색을 수행합니다. `similarity_search` 메서드에 질문을 전달하면, 의미적으로 가장 유사한 문서를 `k`개 반환합니다.

아래 코드는 "LangGraph란 무엇인가?" 질문으로 상위 2개 문서를 검색합니다.

In [4]:
# 검색 테스트
query = "LangGraph란 무엇인가?"
docs = vectorstore.similarity_search(query, k=2)

print(f"Query: {query}\n")
for i, doc in enumerate(docs, 1):
    print(f"Result {i}:")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print()

Query: LangGraph란 무엇인가?

Result 1:
Content: LangGraph는 상태 기반 에이전트를 구축하기 위한 라이브러리입니다. LangChain 위에 구축되었습니다.
Metadata: {'source': 'docs', 'topic': 'langgraph'}

Result 2:
Content: LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다. 도구, 에이전트, 메모리 등을 제공합니다.
Metadata: {'source': 'docs', 'topic': 'langchain'}



## RAG 아키텍처

RAG는 시스템의 필요에 따라 여러 방식으로 구현할 수 있습니다.

| 아키텍처 | 설명 | 제어 | 유연성 | 지연 시간 | 사용 사례 |
|---------|------|------|-------|----------|----------|
| **2-Step RAG** | 검색이 항상 생성 전에 발생. 간단하고 예측 가능 | ✅ 높음 | ❌ 낮음 | ⚡ 빠름 | FAQ, 문서 봇 |
| **Agentic RAG** | LLM 기반 에이전트가 추론 중 검색 시점과 방법을 결정 | ❌ 낮음 | ✅ 높음 | ⏳ 가변적 | 다중 도구를 사용하는 연구 어시스턴트 |
| **Hybrid RAG** | 2-Step RAG와 Agentic RAG의 특성을 결합. 쿼리 전처리, 검색 검증 등 중간 단계 도입 | ⚖️ 중간 | ⚖️ 중간 | ⏳ 가변적 | 품질이 중요한 문서 Q&A, 검색 검증이 필요한 시스템 |

## 2-Step RAG

**2-Step RAG**는 가장 기본적인 RAG 아키텍처로, 검색 단계가 항상 생성 단계 전에 실행됩니다. 사용자 질문이 들어오면 먼저 벡터 저장소에서 관련 문서를 검색하고, 검색된 문서를 컨텍스트로 활용하여 LLM이 답변을 생성합니다. 이 아키텍처는 간단하고 예측 가능하며, LCEL(LangChain Expression Language) 체인으로 구현할 수 있습니다. LangGraph를 활용한 2-Step RAG는 [02-LangGraph-Naive-RAG](02-LangGraph-Naive-RAG.ipynb)에서 자세히 다룹니다.

아래 코드는 Retriever, 프롬프트, LLM을 연결한 2-Step RAG 체인을 구성하고 실행합니다.

In [5]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Retriever 생성
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 프롬프트 템플릿
template = """다음 컨텍스트를 사용하여 질문에 답하세요:

컨텍스트:
{context}

질문: {question}

답변:"""

prompt = ChatPromptTemplate.from_template(template)

# LLM 초기화 (OpenAI 키 사용 시 gpt-4.1-mini 등으로 변경하세요)
llm = init_chat_model("claude-sonnet-4-5", temperature=0)


def format_docs(docs):
    """검색된 문서 리스트를 하나의 문자열로 포맷팅합니다."""
    return "\n\n".join(doc.page_content for doc in docs)


# RAG 체인 구성
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 테스트
question = "LangGraph는 무엇이고 어떻게 사용하나요?"
answer = rag_chain.invoke(question)

print(f"Question: {question}\n")
print(f"Answer: {answer}")

Question: LangGraph는 무엇이고 어떻게 사용하나요?

Answer: # LangGraph란?

**LangGraph**는 **상태 기반 에이전트를 구축하기 위한 라이브러리**입니다.

## 주요 특징

- **LangChain 위에 구축됨**: LangChain 프레임워크를 기반으로 만들어졌습니다
- **상태 기반 접근**: 에이전트의 상태를 관리하며 동작하는 시스템을 구축할 수 있습니다

## LangChain과의 관계

LangGraph는 LangChain과 함께 사용됩니다:
- **LangChain**: LLM 애플리케이션 개발을 위한 기본 프레임워크로, 도구, 에이전트, 메모리 등의 기능을 제공
- **LangGraph**: LangChain 위에서 더 복잡한 상태 기반 에이전트를 구축하기 위한 상위 레이어

---

**참고**: 제공된 컨텍스트에는 LangGraph의 구체적인 사용 방법에 대한 정보가 포함되어 있지 않습니다. 실제 사용법을 알기 위해서는 공식 문서나 추가 자료가 필요합니다.


### 2-Step RAG의 장단점

**장점**:
- 간단하고 예측 가능합니다.
- 응답 시간이 빠릅니다.
- 구현이 쉽습니다.

**단점**:
- 유연성이 부족합니다.
- 불필요한 경우에도 항상 검색을 수행합니다.
- 다중 단계 추론이 제한적입니다.

## Agentic RAG

**Agentic RAG**는 에이전트가 추론 과정에서 **언제** 그리고 **어떻게** 정보를 검색할지 스스로 결정합니다. `create_agent`를 사용하여 검색 도구를 가진 에이전트를 생성하면, LLM이 질문의 특성에 따라 도구 호출 여부를 판단합니다. 외부 정보가 필요하지 않은 질문에는 검색을 수행하지 않아 불필요한 비용을 절감할 수 있습니다.

> **참고 문서**: [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)

아래 코드는 검색 도구를 가진 Agentic RAG 에이전트를 생성하고 테스트합니다.

In [6]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_teddynote.messages import stream_graph

# 검색 도구 생성
@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base for relevant information."""
    docs = vectorstore.similarity_search(query, k=2)
    return "\n\n".join([doc.page_content for doc in docs])

# Agentic RAG 에이전트
agent = create_agent(
    model=llm,
    tools=[search_knowledge_base],
    system_prompt="""당신은 도움이 되는 AI 어시스턴트입니다.
    질문에 답하기 위해 외부 정보가 필요하면 search_knowledge_base 도구를 사용하세요.
    검색 결과를 기반으로 명확하고 정확한 답변을 제공하세요."""
)

# 테스트
stream_graph(
    agent,
    inputs={"messages": [{"role": "user", "content": "RAG에 대해 설명해주세요."}]},
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
RAG는 Retrieval-Augmented Generation의 약자로, 외부 지식을 검색하여 LLM 응답을 향상시킵니다.

LangGraph는 상태 기반 에이전트를 구축하기 위한 라이브러리입니다. LangChain 위에 구축되었습니다.
🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
**RAG(Retrieval-Augmented Generation)**에 대해 설명드리겠습니다.

## RAG란?

**RAG**는 **Retrieval-Augmented Generation**의 약자로, 대규모 언어 모델(LLM)의 응답 품질을 향상시키기 위해 외부 지식을 검색하여 활용하는 기술입니다.

## 주요 특징

1. **외부 지식 활용**: LLM이 학습 데이터에만 의존하지 않고, 실시간으로 외부 데이터베이스나 문서에서 관련 정보를 검색합니다.

2. **정확성 향상**: 최신 정보나 특정 도메인 지식을 검색하여 제공함으로써 더 정확하고 신뢰할 수 있는 답변을 생성합니다.

3. **환각(Hallucination) 감소**: LLM이 잘못된 정보를 만들어내는 현상을 줄이고, 실제 데이터에 기반한 답변을 제공합니다.

## 작동 방식

1. **검색(Retrieval)**: 사용자 질문과 관련된 정보를 외부 지식 베이스에서 검색
2. **증강(Augmentation)**: 검색된 정보를 LLM의 입력에 추가
3. **생성(Generation)**: 증강된 정보를 바탕으로 LLM이 답변 생성

RAG는 특히 최신 정보가 필요하거나, 특정 조직의 내부 문서를 활용해야 하는 경우에 매우 유용합니다.

### Agentic RAG의 장단점

**장점**:
- 유연성이 높습니다.
- 필요할 때만 검색합니다.
- 다중 단계 추론이 가능합니다.
- 여러 도구를 조합할 수 있습니다.

**단점**:
- 지연 시간을 예측하기 어렵습니다.
- LLM 호출이 더 많습니다.
- 구현이 복잡합니다.

### 다중 도구를 사용하는 Agentic RAG

Agentic RAG의 강점은 검색 외에도 다양한 도구를 조합할 수 있다는 점입니다. 아래 예제에서는 지식 베이스 검색, 수학 계산, 현재 날짜 조회 등 세 가지 도구를 에이전트에 제공합니다. 에이전트는 질문을 분석하여 필요한 도구를 적절히 선택하고 조합합니다.

아래 코드는 다중 도구를 가진 에이전트를 생성하고, 복합 질문에 대한 답변을 테스트합니다.

In [7]:
@tool
def calculate(expression: str) -> str:
    """Calculate mathematical expressions."""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_current_date() -> str:
    """Get the current date."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d")

# 여러 도구를 가진 에이전트
multi_tool_agent = create_agent(
    model=llm,
    tools=[search_knowledge_base, calculate, get_current_date],
    system_prompt="""당신은 다재다능한 AI 어시스턴트입니다.
    - 지식 기반 질문: search_knowledge_base 사용
    - 계산: calculate 사용
    - 날짜 정보: get_current_date 사용
    
    적절한 도구를 선택하여 사용자 질문에 답하세요."""
)

# 복합 질문 테스트
questions = [
    "LangChain이 무엇인지 설명하고, 오늘 날짜를 알려주세요.",
    "벡터 데이터베이스에 대해 설명하고, 10 * 25를 계산해주세요.",
]

for question in questions:
    print(f"\n=== Question: {question} ===")
    stream_graph(
        multi_tool_agent,
        inputs={"messages": [{"role": "user", "content": question}]},
    )


=== Question: LangChain이 무엇인지 설명하고, 오늘 날짜를 알려주세요. ===

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
2026-02-28LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다. 도구, 에이전트, 메모리 등을 제공합니다.

LangGraph는 상태 기반 에이전트를 구축하기 위한 라이브러리입니다. LangChain 위에 구축되었습니다.
🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
**LangChain 설명:**

LangChain은 **LLM(대규모 언어 모델) 애플리케이션 개발을 위한 프레임워크**입니다. 주요 특징은 다음과 같습니다:

- **도구(Tools)**: 외부 API나 기능을 LLM과 연결할 수 있습니다
- **에이전트(Agents)**: LLM이 자율적으로 작업을 수행하도록 합니다
- **메모리(Memory)**: 대화 컨텍스트를 유지하고 관리합니다

LangChain을 기반으로 한 **LangGraph**라는 라이브러리도 있는데, 이는 상태 기반 에이전트를 구축하기 위한 도구입니다.

**오늘 날짜:**

오늘은 **2026년 2월 28일**입니다.
=== Question: 벡터 데이터베이스에 대해 설명하고, 10 * 25를 계산해주세요. ===

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
250벡터 데이터베이스는 임베딩을 저장하고 검색하는 데 특화된 데이터베이스입니다. FAISS, Pinecone 등이 있습니다.

LangCha

## Hybrid RAG

**Hybrid RAG**는 2-Step RAG와 Agentic RAG의 특성을 결합한 아키텍처입니다. 단순히 검색 후 생성하는 것이 아니라, **쿼리 향상(Query Enhancement)**, **검색 결과 검증(Retrieval Validation)**, **반복적 쿼리 개선(Iterative Refinement)** 등의 중간 단계를 도입하여 RAG 파이프라인의 품질을 높입니다.

주요 단계는 다음과 같습니다:

1. **쿼리 향상**: 사용자 질문을 더 명확하고 검색에 최적화된 형태로 재작성합니다.
2. **검색**: 향상된 쿼리로 관련 문서를 검색합니다.
3. **검색 검증**: 검색 결과가 질문에 답하기에 충분한지 평가합니다. 불충분하면 쿼리를 개선하여 재검색합니다.
4. **답변 생성**: 검증을 통과한 문서를 기반으로 최종 답변을 생성합니다.

이 패턴은 검색 결과의 품질을 보장하면서도 2-Step RAG보다 높은 정확도를 달성할 수 있습니다. 더 고급 구현 패턴은 [CRAG](07-LangGraph-CRAG.ipynb), [Self-RAG](08-LangGraph-Self-RAG.ipynb), [Adaptive RAG](09-LangGraph-Adaptive-RAG.ipynb) 튜토리얼에서 자세히 다룹니다.

> **참고 문서**: [LangGraph Retrieval](https://docs.langchain.com/oss/python/langchain/retrieval)

아래 코드는 쿼리 향상, 검색 검증, 반복적 개선을 포함하는 Hybrid RAG 파이프라인을 구성합니다.

In [17]:
# 1. 쿼리 향상 프롬프트
query_enhancement_prompt = ChatPromptTemplate.from_template(
    """사용자 질문을 더 명확하고 구체적으로 재작성하세요.
    검색에 최적화된 형태로 만드세요.
    
    원본 질문: {question}
    
    향상된 질문:"""
)

# 2. 검색 검증 프롬프트
validation_prompt = ChatPromptTemplate.from_template(
    """다음 검색 결과가 질문에 답하기에 충분한지 평가하세요.
    
    질문: {question}
    
    검색 결과:
    {results}
    
    충분하면 'YES', 불충분하면 'NO'라고만 답하세요."""
)

# 3. 답변 생성 프롬프트
answer_prompt = ChatPromptTemplate.from_template(
    """검색된 정보를 바탕으로 질문에 답하세요.
    
    질문: {question}
    
    컨텍스트:
    {context}
    
    답변:"""
)


def hybrid_rag(question: str, max_iterations: int = 2) -> str:
    """Hybrid RAG: 쿼리 향상 + 검색 검증 + 반복적 개선을 수행합니다.

    Args:
        question: 사용자 질문
        max_iterations: 최대 검색 반복 횟수
    
    Returns:
        최종 답변 문자열
    """
    # 1단계: 쿼리 향상
    enhanced_query = (query_enhancement_prompt | llm | StrOutputParser()).invoke(
        {"question": question}
    )
    print(f"Enhanced query: {enhanced_query}\n")

    for i in range(max_iterations):
        # 2단계: 검색
        docs = vectorstore.similarity_search(enhanced_query, k=3)
        context = "\n\n".join([doc.page_content for doc in docs])

        print(f"Iteration {i+1}: Retrieved {len(docs)} documents")

        # 3단계: 검색 검증
        validation = (validation_prompt | llm | StrOutputParser()).invoke(
            {"question": question, "results": context}
        )

        if "YES" in validation.upper():
            print("Validation: PASSED\n")
            break
        else:
            print("Validation: FAILED - Refining query\n")
            # 쿼리 재작성
            enhanced_query = f"{enhanced_query} (more specific)"

    # 4단계: 답변 생성
    answer = (answer_prompt | llm | StrOutputParser()).invoke(
        {"question": question, "context": context}
    )

    return answer

### Hybrid RAG 실행

`hybrid_rag` 함수를 실행하여 각 단계의 동작을 확인합니다. 쿼리 향상, 검색 검증, 답변 생성 과정이 순차적으로 진행되며, 검색 결과가 불충분하면 쿼리를 개선하여 재검색합니다.

아래 코드는 Hybrid RAG를 실행하고 최종 답변을 출력합니다.

In [18]:
# Hybrid RAG 테스트
question = "LangChain과 관련된 기술들은?"
answer = hybrid_rag(question)

print(f"\nFinal Answer:\n{answer}")

Enhanced query: # 향상된 질문:

**"LangChain 프레임워크와 함께 사용되는 주요 기술 스택과 통합 가능한 도구들은 무엇인가요? 특히 LLM 모델, 벡터 데이터베이스, 에이전트 프레임워크, 그리고 관련 Python/JavaScript 라이브러리를 포함하여 설명해주세요."**

## 개선 포인트:

1. **구체성 추가**: "관련된 기술들"을 구체적인 카테고리로 세분화
   - LLM 모델
   - 벡터 데이터베이스
   - 에이전트 프레임워크
   - 프로그래밍 라이브러리

2. **맥락 제공**: "프레임워크", "통합 가능한" 등의 키워드로 관계성 명확화

3. **검색 최적화**: 
   - "LangChain"과 함께 검색될 주요 키워드 포함
   - 기술 스택, 도구, 통합 등 검색 친화적 용어 사용

4. **범위 명시**: Python/JavaScript 언어 환경 특정

Iteration 1: Retrieved 3 documents
Validation: PASSED


Final Answer:
LangChain과 관련된 기술들은 다음과 같습니다:

1. **LangGraph**
   - 상태 기반 에이전트를 구축하기 위한 라이브러리
   - LangChain 위에 구축된 기술

2. **벡터 데이터베이스**
   - 임베딩을 저장하고 검색하는 데 특화된 데이터베이스
   - 예시: FAISS, Pinecone

3. **LangChain의 핵심 구성요소**
   - 도구(Tools)
   - 에이전트(Agents)
   - 메모리(Memory)

이러한 기술들은 LLM 애플리케이션 개발 생태계를 구성하며, LangChain을 중심으로 통합되어 활용됩니다.


## 실용적인 예제: 문서 Q&A 시스템

더 많은 문서를 추가하여 확장된 지식 베이스를 구축하고, 이를 기반으로 완전한 문서 Q&A 시스템을 만들어 봅니다. 다양한 주제(프로그래밍, AI, NLP 등)의 문서를 포함하여 실제 환경에 가까운 RAG 시스템을 구현합니다.

아래 코드는 확장된 문서 세트를 생성하고 새로운 벡터 저장소에 저장합니다.

In [10]:
# 더 많은 문서 추가
extended_documents = [
    Document(
        page_content="""Python은 1991년 Guido van Rossum이 개발한 고수준 프로그래밍 언어입니다.
        간결하고 읽기 쉬운 문법으로 유명하며, 데이터 과학, 웹 개발, 자동화 등 다양한 분야에서 사용됩니다.""",
        metadata={"source": "programming", "topic": "python"}
    ),
    Document(
        page_content="""머신러닝은 데이터로부터 학습하는 알고리즘을 연구하는 인공지능의 한 분야입니다.
        지도 학습, 비지도 학습, 강화 학습 등의 방법이 있으며, 이미지 인식, 자연어 처리 등에 활용됩니다.""",
        metadata={"source": "ai", "topic": "machine-learning"}
    ),
    Document(
        page_content="""Transformer는 2017년 Google이 발표한 딥러닝 아키텍처입니다.
        Self-attention 메커니즘을 사용하며, GPT, BERT 등 현대 LLM의 기초가 되었습니다.""",
        metadata={"source": "ai", "topic": "transformer"}
    ),
    Document(
        page_content="""임베딩(Embedding)은 텍스트를 고차원 벡터 공간의 점으로 표현하는 기법입니다.
        의미적으로 유사한 텍스트는 벡터 공간에서 가까이 위치합니다. Word2Vec, BERT embeddings 등이 있습니다.""",
        metadata={"source": "nlp", "topic": "embeddings"}
    ),
]

# 확장된 벡터 저장소 생성
all_documents = documents + extended_documents
extended_vectorstore = FAISS.from_documents(all_documents, embeddings)

print(f"Extended knowledge base with {len(all_documents)} documents")

Extended knowledge base with 8 documents


### 대화형 Q&A 시스템

확장된 지식 베이스를 기반으로 `create_agent`를 사용하여 대화형 Q&A 에이전트를 구현합니다. 에이전트는 사용자 질문을 분석하고, 검색 도구를 활용하여 관련 문서를 찾아 답변을 생성합니다. 출처 토픽도 함께 표시하여 답변의 근거를 제공합니다.

아래 코드는 Q&A 에이전트를 생성하고 다양한 질문으로 테스트합니다.

In [11]:
from langchain.agents import create_agent

@tool
def search_documents(query: str) -> str:
    """Search documents in the knowledge base."""
    docs = extended_vectorstore.similarity_search(query, k=3)
    results = []
    for doc in docs:
        results.append(f"[{doc.metadata['topic']}] {doc.page_content}")
    return "\n\n".join(results)

# 문서 Q&A 에이전트
qa_agent = create_agent(
    model=llm,
    tools=[search_documents],
    system_prompt="""당신은 지식 기반 Q&A 어시스턴트입니다.
    
    사용자 질문에 답할 때:
    1. search_documents 도구를 사용하여 관련 정보를 찾으세요
    2. 검색된 문서를 바탕으로 명확하고 정확한 답변을 제공하세요
    3. 답변의 출처(토픽)를 언급하세요
    4. 검색 결과가 불충분하면 이를 사용자에게 알리세요
    
    한국어로 답변하세요."""
)

# 테스트 질문들
test_questions = [
    "Python에 대해 설명해주세요.",
    "Transformer 아키텍처는 무엇인가요?",
    "임베딩이 무엇이고 어떻게 사용되나요?",
    "RAG와 머신러닝의 관계는?",
]

for question in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    stream_graph(
        qa_agent,
        inputs={"messages": [{"role": "user", "content": question}]},
    )


Q: Python에 대해 설명해주세요.

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
[python] Python은 1991년 Guido van Rossum이 개발한 고수준 프로그래밍 언어입니다.
        간결하고 읽기 쉬운 문법으로 유명하며, 데이터 과학, 웹 개발, 자동화 등 다양한 분야에서 사용됩니다.

[machine-learning] 머신러닝은 데이터로부터 학습하는 알고리즘을 연구하는 인공지능의 한 분야입니다.
        지도 학습, 비지도 학습, 강화 학습 등의 방법이 있으며, 이미지 인식, 자연어 처리 등에 활용됩니다.

[langgraph] LangGraph는 상태 기반 에이전트를 구축하기 위한 라이브러리입니다. LangChain 위에 구축되었습니다.
🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
**Python에 대한 설명**

검색 결과를 바탕으로 Python에 대해 설명드리겠습니다.

## 기본 정보
- **개발자**: Guido van Rossum
- **개발 연도**: 1991년
- **특징**: 고수준 프로그래밍 언어

## 주요 특징
Python은 **간결하고 읽기 쉬운 문법**으로 유명합니다. 이는 초보자도 쉽게 배울 수 있고, 코드의 가독성이 높아 유지보수가 용이하다는 장점이 있습니다.

## 활용 분야
Python은 다양한 분야에서 널리 사용되고 있습니다:
- **데이터 과학**: 데이터 분석, 시각화
- **웹 개발**: 웹 애플리케이션 개발
- **자동화**: 반복 작업 자동화
- **머신러닝/인공지능**: 다양한 AI 프로젝트

Python은 범용성이 뛰어나고 풍부한 라이브러리 생태계를 갖추고 있어, 현재 가장 인기 있는 프로그래밍 언어 중 하나입

## 고급 기능: 메타데이터 필터링

메타데이터를 활용하면 검색 범위를 특정 토픽이나 문서 유형으로 제한하여 더 정확한 결과를 얻을 수 있습니다. FAISS의 `filter` 파라미터를 사용하여 메타데이터 기반 필터링을 적용할 수 있으며, 이를 도구에 통합하면 에이전트가 토픽별로 검색을 수행할 수 있습니다.

아래 코드는 토픽별 문서 검색 도구와 이를 사용하는 에이전트를 구현합니다.

In [12]:
@tool
def search_by_topic(query: str, topic: str) -> str:
    """Search documents filtered by topic.
    
    Available topics: python, machine-learning, transformer, embeddings, langchain, langgraph, rag, vectordb
    """
    # 메타데이터 필터링
    docs = extended_vectorstore.similarity_search(
        query,
        k=3,
        filter={"topic": topic}
    )
    
    if not docs:
        return f"No documents found for topic: {topic}"
    
    results = []
    for doc in docs:
        results.append(doc.page_content)
    return "\n\n".join(results)

# 토픽 필터링 에이전트
filtered_agent = create_agent(
    model=llm,
    tools=[search_by_topic],
    system_prompt="""당신은 토픽별 문서 검색 어시스턴트입니다.
    
    사용자 질문에서 관련 토픽을 파악하고 search_by_topic을 사용하세요.
    가능한 토픽: python, machine-learning, transformer, embeddings, langchain, langgraph, rag, vectordb
    """
)

# 테스트
stream_graph(
    filtered_agent,
    inputs={"messages": [{"role": "user", "content": "Python에 대한 정보를 찾아주세요."}]},
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
Python은 1991년 Guido van Rossum이 개발한 고수준 프로그래밍 언어입니다.
        간결하고 읽기 쉬운 문법으로 유명하며, 데이터 과학, 웹 개발, 자동화 등 다양한 분야에서 사용됩니다.
🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
Python에 대한 정보를 찾았습니다:

**Python 프로그래밍 언어**

- **개발자**: Guido van Rossum
- **출시 연도**: 1991년
- **특징**: 고수준 프로그래밍 언어로, 간결하고 읽기 쉬운 문법으로 유명합니다
- **주요 활용 분야**:
  - 데이터 과학
  - 웹 개발
  - 자동화
  - 그 외 다양한 분야

Python은 초보자부터 전문가까지 널리 사용되는 범용 프로그래밍 언어입니다. 더 구체적인 Python 관련 정보가 필요하시면 말씀해 주세요!

## 모범 사례

### 1. 적절한 청크 크기 선택

청크 크기는 RAG 시스템의 검색 품질에 직접적인 영향을 미칩니다. 너무 작으면 컨텍스트가 부족하고, 너무 크면 노이즈가 증가하여 검색 정확도가 떨어집니다. 일반적으로 500-1000 토큰이 적절합니다.

아래 코드는 권장하는 청크 크기와 오버랩 설정 예시입니다.

In [13]:
# 좋은 예
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # 적절한 크기
    chunk_overlap=50     # 컨텍스트 연속성 유지
)

### 2. 검색 결과 수 조정

검색 결과 수(`k` 값)는 질문의 복잡도에 따라 조정해야 합니다. k 값이 너무 작으면 중요한 정보를 놓칠 수 있고, 너무 크면 노이즈가 증가하고 비용이 높아집니다.

아래 코드는 질문 유형에 따른 k 값 설정 예시입니다.

In [14]:
# 질문 유형에 따라 k 조정
simple_query_k = 2  # 간단한 질문
complex_query_k = 5  # 복잡한 질문

### 3. 프롬프트 최적화

RAG 프롬프트에서는 LLM에게 컨텍스트 활용 방법과 답변 생성 규칙을 명확히 지시해야 합니다. 특히 컨텍스트에 답이 없는 경우의 행동을 정의하면 환각(hallucination)을 줄일 수 있습니다.

아래는 컨텍스트 기반 답변과 정보 부족 시 처리를 포함한 프롬프트 예시입니다.

In [15]:
# 좋은 프롬프트
good_prompt = """다음 컨텍스트를 사용하여 질문에 답하세요.
컨텍스트에 답이 없으면 '정보가 부족합니다'라고 답하세요.
추측하지 마세요.

컨텍스트: {context}
질문: {question}
답변:"""

### 4. 메타데이터 활용

문서에 메타데이터를 풍부하게 추가하면 필터링과 출처 추적이 용이해집니다. 출처, 페이지 번호, 토픽, 작성일, 저자 등의 정보를 포함시키는 것을 권장합니다.

아래는 유용한 메타데이터 필드의 예시입니다.

In [16]:
# 유용한 메타데이터 필드 예시
metadata = {
    "source": "document_name.pdf",
    "page": 5,
    "topic": "machine-learning",
    "date": "2024-01-01",
    "author": "John Doe",
}

## 정리

이 튜토리얼에서는 RAG의 세 가지 아키텍처를 비교하고, 각각의 구현 방법을 살펴보았습니다.

### RAG 아키텍처 선택 가이드

**2-Step RAG를 선택하는 경우**:
- 간단한 FAQ 시스템
- 예측 가능한 지연 시간이 중요할 때
- 구현 복잡도를 최소화하고 싶을 때

**Agentic RAG를 선택하는 경우**:
- 복잡한 다단계 추론 필요
- 여러 도구와 데이터 소스 활용
- 유연성이 중요할 때

**Hybrid RAG를 선택하는 경우**:
- 검색 결과의 품질 검증이 필요할 때
- 쿼리 전처리로 검색 정확도를 높이고 싶을 때
- 2-Step RAG보다 높은 품질이 필요하지만 Agentic RAG만큼의 유연성은 불필요할 때

### 핵심 요점

1. **지식 베이스 구축**: 문서 로드 → 청크 분할 → 임베딩 → 벡터 저장소
2. **검색**: 쿼리 임베딩 → 유사도 검색 → 관련 문서 반환
3. **생성**: 검색된 컨텍스트 + 질문 → LLM → 답변
4. **Hybrid RAG**: 쿼리 전처리 + 검색 검증 + 조건부 생성 (LangGraph 활용)
5. **최적화**: 청크 크기, k 값, 프롬프트, 메타데이터 활용

RAG는 LLM의 지식을 확장하고 최신 정보를 제공하는 강력한 패턴입니다.

### 추가 학습 자료

더 깊이 있는 학습을 원한다면 아래 공식 문서와 관련 튜토리얼을 참고하세요:

- [LangChain Retrieval](https://docs.langchain.com/oss/python/langchain/retrieval)
- [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents.md)
- [LangGraph Agentic RAG](https://docs.langchain.com/oss/python/langgraph/agentic-rag)
- [CRAG 튜토리얼](07-LangGraph-CRAG.ipynb) / [Self-RAG 튜토리얼](08-LangGraph-Self-RAG.ipynb) / [Adaptive RAG 튜토리얼](09-LangGraph-Adaptive-RAG.ipynb)